# LightGBM + Meta-Labeling

Sprint 1 из дорожной карты: meta-labeling поверх LightGBM Primary.

## Что было

После R-0054 fix чистый best Sharpe = **1.14** (h=12 max_pnl, n=15 сделок).
Цифра нестабильная: 15 сделок — статистический шум.

## Идея meta-labeling (Lopez de Prado, AFML 2018, ch. 3)

Primary отвечает: «направление вверх или вниз?» → `P(UP)`
Meta отвечает: «стоит ли действовать на этом сигнале Primary?» → `P(profit | signal)`

Вход Meta: **prediction Primary** + контекст (volatility, spread, регим).
Выход: вероятность прибыльной сделки. Торгуем когда Primary > T1 AND Meta > T2.

## Ожидаемый эффект

Sharpe 1.14 → **1.5–1.7** при сохранении/росте n_BUY (более стабильная статистика).

## Архитектура

1. Primary OOF (5-fold TimeSeriesSplit) → out-of-sample predictions на train.
2. Final Primary обучается на полном train для inference.
3. Meta features = primary OOF + ~10 контекстных признаков.
4. Meta target = `1 if lr > 2·cost else 0` (margin of safety 2× costs).
5. Meta = LightGBM на meta features.
6. На test: Primary predict → Meta predict → final signal.

Время: ~10–15 минут на Colab CPU.

## 0. Окружение

In [ ]:
from __future__ import annotations

import os, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy(); env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH], check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True, env=env)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
                        f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
                       check=True, env=env)
else:
    PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

In [ ]:
if IN_COLAB:
    subprocess.run(['pip', 'install', '--quiet',
                    'pandas>=2.1', 'pyarrow>=15.0', 'scikit-learn>=1.4',
                    'lightgbm>=4.0', 'tqdm>=4.66', 'matplotlib>=3.7'],
                   check=True)
    print('Deps OK')

In [ ]:
import sys
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from graduate_work.config import default_config

TICKERS = (
    'SBER', 'VTBR', 'GAZP', 'LKOH', 'GMKN', 'ROSN', 'NVTK',
    'MGNT', 'TATN', 'MTSS', 'MOEX', 'NLMK', 'CHMF', 'ALRS',
)

cfg = default_config()
cfg = dataclasses.replace(cfg, data=dataclasses.replace(cfg.data, tickers=TICKERS))
print(f'tickers={cfg.data.tickers}\nhorizons={cfg.data.horizons}')

## 1. Drive: данные ALGOPACK

In [ ]:
import shutil
DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    src_dir = DRIVE_BASE / 'data' / 'raw' / 'Algopack'
    if not src_dir.exists():
        src_dir = DRIVE_BASE / 'data' / 'raw'
    if src_dir.exists():
        shutil.copytree(src_dir, cfg.paths.data_raw, dirs_exist_ok=True)
        print(f'Скопировано из {src_dir}')

for sub in ['algopack/tradestats', 'algopack/orderstats', 'algopack/obstats', 'algopack/hi2']:
    p = cfg.paths.data_raw / sub
    if p.exists():
        n = sum(1 for _ in p.glob('*.csv'))
        print(f'  {sub}: {n} файлов')

## 2. Загрузка + Per-ticker features + Cross-sectional

In [ ]:
from graduate_work.features.algopack_features import (
    obstats_features, orderstats_features, tradestats_features,
    order_to_trade_ratio, hi2_features,
)
from graduate_work.features.targets import cost_aware_classification_labels, lr_columns
from graduate_work.features.cross_sectional import add_cross_sectional_features


def _load_csv_indexed(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    if df.empty or 'tradedate' not in df.columns:
        return df
    if 'tradetime' in df.columns:
        ts = pd.to_datetime(df['tradedate'].astype(str) + ' ' + df['tradetime'].astype(str),
                            utc=True, errors='coerce')
    else:
        ts = pd.to_datetime(df['tradedate'], utc=True, errors='coerce')
    df.index = ts
    df.index.name = 'begin'
    return df.dropna(how='all').sort_index()


ts_data, os_data, ob_data, hi2_data = {}, {}, {}, {}
for ticker in TICKERS:
    ts_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'tradestats' / f'{ticker}.csv')
    os_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'orderstats' / f'{ticker}.csv')
    ob_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'obstats' / f'{ticker}.csv')
    hi2_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'hi2' / f'{ticker}.csv')

loaded = {t: len(ts_data[t]) for t in TICKERS if not ts_data[t].empty}
missing = set(TICKERS) - set(loaded.keys())
if missing:
    print(f'MISSING: {sorted(missing)}')
    TICKERS = tuple(t for t in TICKERS if t not in missing)


def _ts_to_ohlcv(ts: pd.DataFrame, ticker: str) -> pd.DataFrame:
    out = pd.DataFrame(index=ts.index)
    out['open']   = ts['pr_open'].astype(float)
    out['high']   = ts['pr_high'].astype(float)
    out['low']    = ts['pr_low'].astype(float)
    out['close']  = ts['pr_close'].astype(float)
    out['volume'] = ts['vol'].astype(float)
    out['value']  = ts['val'].astype(float)
    out['vwap']   = ts['pr_vwap'].astype(float)
    for col in ['sec_pr_open', 'sec_pr_high', 'sec_pr_low', 'sec_pr_close']:
        if col in ts.columns:
            out[col] = ts[col].astype(float)
    out['ticker'] = ticker
    return out


def _build_log_returns(close: pd.Series) -> pd.DataFrame:
    lr = np.log(close / close.shift(1))
    return pd.DataFrame({
        'lr_1':  lr,
        'lr_5':  lr.rolling(5).sum(),
        'lr_15': lr.rolling(15).sum(),
        'lr_30': lr.rolling(30).sum(),
    }, index=close.index)


def _build_sec_features(feat: pd.DataFrame) -> pd.DataFrame:
    if 'sec_pr_close' not in feat.columns:
        return pd.DataFrame(index=feat.index)
    out = pd.DataFrame(index=feat.index)
    sec_close = feat['sec_pr_close'].astype(float)
    out['sec_rel_strength'] = (feat['close'] / sec_close.replace(0, np.nan)).fillna(1.0) - 1.0
    sec_lr = np.log(sec_close / sec_close.shift(1))
    out['sec_lr_1'] = sec_lr.fillna(0.0)
    out['sec_lr_5'] = sec_lr.rolling(5).sum().fillna(0.0)
    out['sec_lr_15'] = sec_lr.rolling(15).sum().fillna(0.0)
    own_lr = np.log(feat['close'] / feat['close'].shift(1))
    out['sec_lr_residual'] = (own_lr - sec_lr).fillna(0.0)
    return out


def build_per_ticker(ticker: str) -> pd.DataFrame:
    ts_df = ts_data[ticker]
    if ts_df.empty:
        return pd.DataFrame()
    feat = _ts_to_ohlcv(ts_df, ticker)
    feat = feat.join(_build_log_returns(feat['close']), how='left')
    feat = feat.join(_build_sec_features(feat), how='left')
    feat = feat.join(tradestats_features(ts_df), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(orderstats_features(os_data[ticker]), how='left')
    if not ob_data[ticker].empty:
        feat = feat.join(obstats_features(ob_data[ticker]), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(order_to_trade_ratio(os_data[ticker], ts_data[ticker]), how='left')
    if not hi2_data[ticker].empty:
        feat = feat.join(hi2_features(hi2_data[ticker], feat.index), how='left')
    return feat.fillna(0.0)


per_ticker = {t: build_per_ticker(t) for t in TICKERS}
per_ticker = {t: df for t, df in per_ticker.items() if not df.empty}

# Cross-sectional
XS_FEATURES = [
    'aps_vol_imb', 'aps_disb', 'aps_vwap_premium',
    'aps_imb_vol_bbo', 'aps_pr_change_bp', 'aps_spread_bbo_bp',
]
per_ticker_xs = add_cross_sectional_features(
    per_ticker, base_features=XS_FEATURES,
    rank=True, zscore=True, relative_mode='diff',
)
print(f'Per-ticker frames: {len(per_ticker_xs)}; sample shape: {next(iter(per_ticker_xs.values())).shape}')

## 3. Merge + targets + chronological split

In [ ]:
full = pd.concat(per_ticker_xs.values()).sort_index()

out_parts = []
costs = cfg.trading.commission_rate + cfg.trading.slippage_rate
for ticker, sub in full.groupby('ticker'):
    labels = cost_aware_classification_labels(
        open_price=sub['open'], close_price=sub['close'],
        horizons=cfg.data.horizons,
        entry_cost=costs, exit_cost=costs,
        label_smoothing=0.0, direction='long',
    )
    out_parts.append(sub.join(labels, how='left'))
full_with_targets = pd.concat(out_parts).sort_values(['ticker']).sort_index(kind='stable')

target_cols = [f'target_h{h}' for h in cfg.data.horizons]
lr_cols_list = lr_columns(cfg.data.horizons)
feature_cols = [c for c in full_with_targets.columns
                if c not in ('ticker',) and c not in target_cols and c not in lr_cols_list]
print(f'Multi-ticker frame: {full_with_targets.shape}, features: {len(feature_cols)}')

from graduate_work.features.pipeline import chronological_split
train_df, val_df, test_df = chronological_split(
    full_with_targets,
    cfg.data.train_ratio, cfg.data.val_ratio,
    purge_horizon=max(cfg.data.horizons),
)
test_prices_raw = test_df[['open', 'close', 'ticker']].copy()
print(f'train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')

## 4. Meta-Labeling Pipeline

**Primary features** = все 90+ микроструктурных фич (как раньше).
**Meta features** = primary's predictions + ~15 контекстных фич.

Контекстные фичи — выбраны так, чтобы НЕ дублировать primary, а
характеризовать **режим**: волатильность, спред, регим (HI2),
недавнее momentum. Это даёт meta-у новое зрение, которого не было у primary.

In [ ]:
from graduate_work.model.lgbm_pipeline import LightGBMConfig
from graduate_work.model.meta_labeling import MetaLabelingPipeline

# Meta context features — outline of regime + Primary's score(s)
META_CONTEXT_FEATURES = [
    # Volatility & liquidity
    'aps_intra_vol_bp', 'aps_spread_bbo_bp', 'aps_spread_lv10_bp',
    # Recent momentum
    'lr_5', 'lr_15', 'lr_30',
    # Order-book regime
    'aps_levels_total', 'aps_imb_vol_bbo', 'aps_levels_imb',
    # Daily regim — HI2 (после R-0054 уже не leakage)
    'aps_hi2_hhi_volume', 'aps_hi2_hhi_aggressive', 'aps_hi2_hhi_passive',
    # Sectoral context
    'sec_rel_strength', 'sec_lr_residual',
    # Cross-sectional rank — где этот тикер относительно universe
    'aps_imb_vol_bbo_xrank', 'aps_disb_xrank',
]
# Добавить primary_h{h} per horizon (создаются автоматически в pipeline)
META_CONTEXT_FEATURES = [c for c in META_CONTEXT_FEATURES if c in feature_cols]
META_FEATURES = META_CONTEXT_FEATURES + [f'primary_h{h}' for h in cfg.data.horizons]

primary_cfg = LightGBMConfig(
    n_estimators=200, num_leaves=31, learning_rate=0.05,
    min_child_samples=200, reg_lambda=1.0, reg_alpha=0.1,
    early_stopping_rounds=30, random_state=42,
)
meta_cfg = LightGBMConfig(
    n_estimators=300, num_leaves=15, learning_rate=0.03,
    min_child_samples=500, reg_lambda=2.0, reg_alpha=0.5,  # сильнее регуляризован
    early_stopping_rounds=30, random_state=42,
)

pipeline = MetaLabelingPipeline(
    horizons=cfg.data.horizons,
    primary_features=feature_cols,
    meta_features=META_FEATURES,
    primary_cfg=primary_cfg,
    meta_cfg=meta_cfg,
    cost_per_trade=costs,
    profit_multiplier=2.0,  # требуем lr > 2·cost для meta_target=1
    n_oof_splits=5,
)
summary = pipeline.fit(train_df, val_df)

ckpt_dir = cfg.paths.checkpoints / 'meta_labeling'
ckpt_dir.mkdir(parents=True, exist_ok=True)
pipeline.primary.save(ckpt_dir / 'primary')
pipeline.meta.save(ckpt_dir / 'meta')

print('\n=== Per-horizon Primary results ===')
for h, m in summary['primary'].items():
    print(f'  h={h}: best_iter={m["best_iter"]}, val_log_loss={m["val_log_loss"]:.4f}, val_auc={m["val_auc"]:.4f}')
print('\n=== Per-horizon Meta results ===')
for h, m in summary['meta'].items():
    print(f'  h={h}: best_iter={m["best_iter"]}, val_log_loss={m["val_log_loss"]:.4f}, val_auc={m["val_auc"]:.4f}')

## 5. Predict on test + diagnostic

In [ ]:
primary_preds, meta_preds = pipeline.predict(test_df)
primary_preds['timestamp'] = pd.to_datetime(primary_preds['timestamp'], utc=True)
meta_preds['timestamp'] = pd.to_datetime(meta_preds['timestamp'], utc=True)

test_long_index = pd.to_datetime(test_df.index, utc=True)

print('=== Per-horizon: Primary AUC vs Meta AUC on test ===')
print(f'{"horizon":>8} | {"P(UP)":>7} | {"prim mean":>9} | {"meta mean":>9} | {"prim AUC":>9} | {"meta AUC":>9} | {"n":>7}')
for h in cfg.data.horizons:
    target_col = f'target_h{h}'
    lr_col = f'lr_h{h}'
    if target_col not in test_df.columns:
        continue
    mask = test_df[target_col].notna()
    if not mask.any():
        continue
    sub = pd.DataFrame({
        'timestamp': test_long_index[mask.to_numpy()],
        'ticker': test_df.loc[mask, 'ticker'].to_numpy(),
        'target': test_df.loc[mask, target_col].astype(int).to_numpy(),
        'lr': test_df.loc[mask, lr_col].to_numpy(),
    })
    h_prim = primary_preds[primary_preds['horizon'] == h]
    h_meta = meta_preds[meta_preds['horizon'] == h]
    merged_p = h_prim.merge(sub, on=['timestamp', 'ticker'], how='inner')
    merged_m = h_meta.merge(sub, on=['timestamp', 'ticker'], how='inner')
    p_up = float(merged_p['target'].mean())
    auc_prim = roc_auc_score(merged_p['target'], merged_p['mean']) if merged_p['target'].nunique() > 1 else float('nan')
    # Meta AUC: predict P(profit), target = (lr > 2·cost)
    meta_target = (merged_m['lr'] > 2 * costs).astype(int)
    auc_meta = roc_auc_score(meta_target, merged_m['mean']) if meta_target.nunique() > 1 else float('nan')
    print(f'{h:>8} | {p_up:>7.3f} | {merged_p["mean"].mean():>9.4f} | {merged_m["mean"].mean():>9.4f} | '
          f'{auc_prim:>9.4f} | {auc_meta:>9.4f} | {len(merged_p):>7d}')

print('\nЕсли meta AUC выше primary AUC — meta улавливает дополнительный сигнал')
print('(прибыльность ≠ просто направление). Если ниже — meta не помогает.')

## 6. Strategy comparison: Primary alone vs Primary × Meta

Сравнение:
- **Primary-only** (как в lightgbm_only.ipynb): max_pnl threshold на primary.
- **Meta-filtered**: торгуем когда `primary > T1 AND meta > T2`.

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest
from graduate_work.strategy import (
    max_pnl_threshold, signals_per_horizon_threshold, top_k_threshold,
)

bars_per_year = cfg.data.bars_per_year
cost_per_trade = 2.0 * costs

# Long-form lr-targets для max_pnl на val
val_lr_rows = []
for h in cfg.data.horizons:
    lr_col = f'lr_h{h}'
    if lr_col not in val_df.columns:
        continue
    sub = val_df[val_df[f'target_h{h}'].notna() & val_df[lr_col].notna()]
    for ts, row in sub.iterrows():
        val_lr_rows.append({
            'timestamp': pd.to_datetime(ts, utc=True), 'ticker': row['ticker'],
            'horizon': int(h), 'actual': float(row[lr_col]),
        })
val_lr_targets = pd.DataFrame(val_lr_rows)
if not val_lr_targets.empty:
    val_lr_targets['timestamp'] = pd.to_datetime(val_lr_targets['timestamp'], utc=True)

val_primary_preds, val_meta_preds = pipeline.predict(val_df)
val_primary_preds['timestamp'] = pd.to_datetime(val_primary_preds['timestamp'], utc=True)
val_meta_preds['timestamp'] = pd.to_datetime(val_meta_preds['timestamp'], utc=True)


def _bt(signals: pd.DataFrame) -> dict:
    n_buy = int((signals['action'] == 'BUY').sum())
    if n_buy == 0:
        return {'n_buy': 0, 'sharpe': 0.0, 'total_return': 0.0,
                'win_rate': 0.0, 'max_dd': 0.0}
    bt = run_backtest(signals, test_prices_raw, cfg.trading)
    m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
    return {'n_buy': n_buy, 'sharpe': float(m['sharpe']),
            'total_return': float(m['total_return']),
            'win_rate': float(m['win_rate']),
            'max_dd': float(m['max_drawdown'])}


rows = []
for h in cfg.data.horizons:
    if h not in pipeline.primary.models or h not in pipeline.meta.models:
        continue
    h_val_prim = val_primary_preds[val_primary_preds['horizon'] == h]
    h_val_meta = val_meta_preds[val_meta_preds['horizon'] == h]
    h_val_lr = val_lr_targets[val_lr_targets['horizon'] == h]

    # === Primary-only baseline (без meta-фильтра) ===
    # max_pnl threshold на val
    if not h_val_lr.empty:
        merged = h_val_prim.merge(h_val_lr, on=['timestamp', 'ticker', 'horizon'], how='inner')
        T_prim, _ = max_pnl_threshold(
            merged['mean'].to_numpy(), merged['actual'].to_numpy(),
            cost_per_trade=cost_per_trade, min_trades=50,
        ) if not merged.empty else (0.5, [])
    else:
        T_prim = 0.5
    sig = signals_per_horizon_threshold(primary_preds, h, threshold=T_prim)
    rows.append({'horizon': h, 'strategy': 'primary_max_pnl',
                 'T_prim': T_prim, 'T_meta': None, **_bt(sig)})

    # === Meta-filtered: primary > T1 AND meta > T2 ===
    # T1 = primary threshold (max_pnl from val)
    # T2 = meta threshold — sweep по нескольким значениям
    for T_meta in [0.5, 0.55, 0.6, 0.65, 0.7]:
        # Combine primary + meta predictions per (timestamp, ticker)
        merged_pm = primary_preds[primary_preds['horizon'] == h].merge(
            meta_preds[meta_preds['horizon'] == h],
            on=['timestamp', 'ticker', 'horizon'],
            suffixes=('_prim', '_meta'),
        )
        # Filter: primary > T_prim AND meta > T_meta
        ok = (merged_pm['mean_prim'] > T_prim) & (merged_pm['mean_meta'] > T_meta)
        merged_filt = merged_pm[ok].copy()
        # Use 'mean_prim' as final 'mean' for signals_per_horizon
        merged_filt['mean'] = merged_filt['mean_prim']
        merged_filt['std'] = 0.0
        sig = signals_per_horizon_threshold(merged_filt, h, threshold=T_prim - 0.01)
        rows.append({
            'horizon': h, 'strategy': f'meta_filt_T{T_meta:.2f}',
            'T_prim': T_prim, 'T_meta': T_meta, **_bt(sig),
        })

results = pd.DataFrame(rows)
print('=== Strategy × horizon matrix (Sharpe) ===')
pivot_sharpe = results.pivot_table(index='strategy', columns='horizon', values='sharpe')
print(pivot_sharpe.round(3).to_string())

print('\n=== n_BUY matrix ===')
pivot_nbuy = results.pivot_table(index='strategy', columns='horizon', values='n_buy', aggfunc='first')
print(pivot_nbuy.astype(int).to_string())

print('\n=== win_rate matrix ===')
pivot_wr = results.pivot_table(index='strategy', columns='horizon', values='win_rate')
print(pivot_wr.round(3).to_string())

print('\n=== total_return % matrix ===')
pivot_tr = results.pivot_table(index='strategy', columns='horizon', values='total_return') * 100
print(pivot_tr.round(2).to_string())

## 7. Verdict + сравнение с Primary-only baseline

In [ ]:
positive = results[results['sharpe'] > 0].sort_values('sharpe', ascending=False)
if not positive.empty:
    print('=== TOP POSITIVE-SHARPE COMBINATIONS ===')
    print(positive.head(10).to_string(
        index=False,
        float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x),
    ))
    best = positive.iloc[0]
    print(f'\nBEST: {best["strategy"]} @ h={best["horizon"]}: '
          f'Sharpe={best["sharpe"]:.3f}, return={best["total_return"]*100:.2f}%, '
          f'n_BUY={int(best["n_buy"])}, win_rate={best["win_rate"]:.3f}')
    # Сравнение с Primary-only baseline (max_pnl)
    primary_only = results[results['strategy'] == 'primary_max_pnl']
    if not primary_only.empty:
        best_primary = primary_only.sort_values('sharpe', ascending=False).iloc[0]
        print(f'\nСравнение: meta-filter vs primary-only')
        print(f'  Primary-only best: {best_primary["strategy"]} @ h={best_primary["horizon"]} → '
              f'Sharpe={best_primary["sharpe"]:.3f}, n={int(best_primary["n_buy"])}')
        if best['strategy'] != 'primary_max_pnl':
            improvement = (best['sharpe'] - best_primary['sharpe']) / max(abs(best_primary['sharpe']), 1e-6)
            print(f'  Meta-filter improvement: +{improvement*100:.1f}% Sharpe')
else:
    print('=== Все стратегии негативные ===')
    least_bad = results.sort_values('sharpe', ascending=False).iloc[0]
    print(f'Наименее плохая: {least_bad["strategy"]} @ h={least_bad["horizon"]} → Sharpe={least_bad["sharpe"]:.3f}')

## 8. Визуализация: Primary-only vs Meta-filtered + feature importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ax = axes[0]
pivot_sharpe.plot.bar(ax=ax)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(1.14, color='green', ls='--', lw=0.8, label='Primary-only best (1.14)')
ax.set_title('Sharpe per (strategy × horizon)')
ax.set_ylabel('Sharpe (test)')
ax.legend()
ax.grid(alpha=0.3)

# Meta feature importance для лучшего горизонта
ax = axes[1]
best_h = positive.iloc[0]['horizon'] if not positive.empty else cfg.data.horizons[-1]
meta_model = pipeline.meta.models[best_h].model
fi = pd.DataFrame({
    'feature': META_FEATURES,
    'importance': meta_model.feature_importances_,
}).sort_values('importance', ascending=False)
ax.barh(fi['feature'][::-1], fi['importance'][::-1])
ax.set_title(f'Meta feature importance (h={int(best_h)})')
ax.set_xlabel('importance')
plt.tight_layout(); plt.show()

# Сравнение распределений Primary и Meta
fig, axes = plt.subplots(1, len(cfg.data.horizons), figsize=(16, 3.5), sharey=True)
if len(cfg.data.horizons) == 1:
    axes = [axes]
for ax, h in zip(axes, cfg.data.horizons):
    pp = primary_preds[primary_preds['horizon'] == h]['mean'].values
    mp = meta_preds[meta_preds['horizon'] == h]['mean'].values
    ax.hist(pp, bins=40, alpha=0.5, label='Primary', color='steelblue')
    ax.hist(mp, bins=40, alpha=0.5, label='Meta', color='darkorange')
    ax.set_title(f'h={h}')
    ax.set_xlabel('predicted prob')
    ax.legend()
axes[0].set_ylabel('count')
plt.suptitle('Primary vs Meta prediction distributions')
plt.tight_layout(); plt.show()

# Diagnostic: сколько % primary-сигналов «убил» meta-filter
print('\nMeta filter diagnostic:')
for h in cfg.data.horizons:
    if h not in pipeline.primary.models:
        continue
    primary_only_row = results[(results['strategy'] == 'primary_max_pnl') &
                                (results['horizon'] == h)]
    if primary_only_row.empty:
        continue
    n_prim = int(primary_only_row.iloc[0]['n_buy'])
    print(f'  h={h}: Primary-alone trades={n_prim}, Meta-filter уменьшает по T_meta:')
    for T in [0.5, 0.55, 0.6, 0.65, 0.7]:
        meta_row = results[(results['strategy'] == f'meta_filt_T{T:.2f}') &
                            (results['horizon'] == h)]
        if not meta_row.empty:
            n_meta = int(meta_row.iloc[0]['n_buy'])
            sharpe_meta = float(meta_row.iloc[0]['sharpe'])
            kept_pct = 100 * n_meta / max(n_prim, 1)
            print(f'    T_meta={T:.2f}: kept {n_meta} ({kept_pct:.0f}%), Sharpe={sharpe_meta:.3f}')